# Fase 3 (ROP_paper) en Colab - fine-tuning end-to-end con cap=8 / batch_size=8 completos

**Por que este notebook existe:** la corrida local (RTX 5070 Laptop, 8GB VRAM) no sostiene de forma confiable `batch_size=8` + `MAX_BAG_SIZE=8` con el backbone descongelado (verificado con un OutOfMemoryError real) - la composicion de cada batch es aleatoria, asi que a veces entra y a veces no. Este notebook corre el **mismo codigo sin cambios** (`system/training/phase3_finetune/`) en una GPU de Colab con mas VRAM.

**Antes de correr esto, en tu Drive necesitas subir** (una carpeta, cualquier nombre - abajo se asume `MyDrive/ROP_paper_colab_data/`). Ya la tenes armada en `datasets/colab_upload/` localmente - subi esa carpeta entera tal cual (los datasets quedaron comprimidos en .zip a proposito, mucho mas rapido/confiable de subir que miles de archivos sueltos - este notebook los descomprime del lado de Colab):
```
ROP_paper_colab_data/
├── RIDIRP.zip            (2.61GB, 6.004 archivos adentro)
├── FARFUM.zip             (2.16GB, 1.533 archivos adentro)
├── ROP-VL.zip              (0.70GB, 2.022 archivos adentro)
├── manifest/
│   └── rop_unified_manifest.parquet
├── checkpoints_phase1/
│   └── backbone_best.pt   (329MB)
└── checkpoints_phase2/
    ├── fold0_best.pt
    ├── fold1_best.pt
    ├── fold2_best.pt
    ├── fold3_best.pt
    └── fold4_best.pt       (18MB los 5 juntos)
```
Total ≈ 5.86GB, solo 10 archivos (en vez de ~9.560 sueltos). **No hace falta** `HVDROPDB` ni `cached_tokens` (eran solo para Fase 1/2).

**Antes de correr esto, en tu PC:** hace falta que el codigo este pusheado al repo de GitHub (`JuanSho780/ROP_paper`), incluyendo todo `system/training/` (common/, phase1_segmentation/, phase2_warmup/, phase3_finetune/) y `requirements.txt` - este notebook lo clona directo, no hay que subirlo a mano.

## 0. Verificar GPU asignada

Entorno de ejecucion (arriba a la derecha) -> cambiar tipo de entorno -> GPU. Cualquier GPU de Colab (T4, L4, A100) tiene mas VRAM que los 8GB locales.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 1. Montar Google Drive

Te va a pedir autorizar acceso - es necesario para leer `ROP_paper_colab_data/` y, al final, para guardar los checkpoints resultantes de vuelta (el disco local de Colab se borra cuando se cierra la sesion).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/ROP_paper_colab_data'  # ajustar si usaste otro nombre/ruta
import os
assert os.path.isdir(DRIVE_DATA_DIR), f'No se encontro {DRIVE_DATA_DIR} - revisa que subiste la carpeta con ese nombre exacto a tu Drive'
print('OK, Drive montado y carpeta de datos encontrada:', DRIVE_DATA_DIR)

## 2. Clonar/actualizar el repo (el codigo, no los datos)

El `system/training/phase3_finetune/` que trae este paso es el mismo que corre localmente - sin cambios. **No borra el proyecto si ya existe** (ver celda de abajo) - si lo hiciera, cualquier checkpoint de Fase 3 que ya se hubiera generado localmente en esta sesion (y que todavia no copiaste a Drive con el paso 7) se perderia antes de poder reanudar. Si hiciste commits nuevos, asegurate de haberlos pusheado antes de correr esta celda - se trae lo ultimo de GitHub de todas formas.

In [ ]:
import os
import shutil

PROJECT_DIR = '/content/ROP_paper'
PHASE3_CKPT_DIR = f'{PROJECT_DIR}/system/training/phase3_finetune/checkpoints'
BACKUP_DIR = '/content/_phase3_checkpoints_backup'

# Respaldamos cualquier checkpoint de Fase 3 que ya exista ANTES de tocar el
# repo, sin importar que rama del codigo de abajo termine corriendo - asi
# nunca se pierde una corrida en progreso, pase lo que pase con el clonado.
if os.path.isdir(PHASE3_CKPT_DIR) and os.listdir(PHASE3_CKPT_DIR):
    print('Respaldando checkpoints de Fase 3 existentes antes de actualizar el repo...')
    shutil.copytree(PHASE3_CKPT_DIR, BACKUP_DIR, dirs_exist_ok=True)

if os.path.isdir(f'{PROJECT_DIR}/.git'):
    # ya es un clon valido (ej. reconectando tras una desconexion) - actualiza
    # el codigo sin tocar datasets/ ni checkpoints/ (git pull no los toca,
    # ambos estan en .gitignore).
    %cd {PROJECT_DIR}
    !git pull
elif os.path.isdir(PROJECT_DIR):
    # existe pero NO es un repo git (ej. quedo a medio clonar en un intento
    # anterior, o solo tiene los datasets ya descomprimidos) - clonamos aparte
    # y reemplazamos, en vez de un "rm -rf" a ciegas sobre lo que haya adentro.
    !rm -rf /content/ROP_paper_fresh_clone
    !git clone https://github.com/JuanSho780/ROP_paper.git /content/ROP_paper_fresh_clone
    !rm -rf {PROJECT_DIR}
    !mv /content/ROP_paper_fresh_clone {PROJECT_DIR}
    %cd {PROJECT_DIR}
else:
    !git clone https://github.com/JuanSho780/ROP_paper.git {PROJECT_DIR}
    %cd {PROJECT_DIR}

# Restauramos el respaldo de checkpoints de Fase 3, si habia.
if os.path.isdir(BACKUP_DIR):
    os.makedirs(PHASE3_CKPT_DIR, exist_ok=True)
    shutil.copytree(BACKUP_DIR, PHASE3_CKPT_DIR, dirs_exist_ok=True)
    print('Checkpoints de Fase 3 restaurados tras actualizar el repo.')

!git log -1 --format='%H %ci %s'

## 3. Instalar dependencias

Colab ya trae `torch`/`torchvision` con CUDA preinstalado y emparejado con su propio driver - **no los reinstalamos** (forzar la version `+cu128` que usamos localmente podria romper la build de Colab, que usa otra version de CUDA). Solo se instala lo que falta.

In [ ]:
!pip install -q timm==1.0.30 albumentations==2.0.8 opencv-python-headless

import torch
print('torch:', torch.__version__, '| cuda disponible:', torch.cuda.is_available())
import timm, albumentations
print('timm:', timm.__version__, '| albumentations:', albumentations.__version__)

## 4. Descomprimir/copiar datos de Drive al disco local de Colab

**Por que copiar y no leer directo desde Drive montado:** Drive montado usa un sistema de archivos tipo red (FUSE) - leer miles de imagenes individuales desde ahi en cada epoca es notablemente mas lento que disco local. Se descomprime una vez al arrancar la sesion (unos minutos, ~5.86GB comprimidos) y despues el entrenamiento lee del disco local rapido de Colab. El disco local es efimero (se borra al cerrar la sesion) - por eso los checkpoints resultantes se copian de vuelta a Drive al final (paso 7).

In [ ]:
import shutil
import time
import zipfile

t0 = time.time()

extracted_dst = f'{PROJECT_DIR}/datasets/extracted'
os.makedirs(extracted_dst, exist_ok=True)
for name in ['RIDIRP', 'FARFUM', 'ROP-VL']:
    if os.path.isdir(f'{extracted_dst}/{name}'):
        print(f'{name}: ya estaba descomprimido, se omite (reconexion)')
        continue
    zip_path = f'{DRIVE_DATA_DIR}/{name}.zip'
    print(f'descomprimiendo {name}...')
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extracted_dst)  # el zip ya trae el prefijo "{name}/" adentro

os.makedirs(f'{PROJECT_DIR}/datasets/manifest', exist_ok=True)
shutil.copy(f'{DRIVE_DATA_DIR}/manifest/rop_unified_manifest.parquet', f'{PROJECT_DIR}/datasets/manifest/')

os.makedirs(f'{PROJECT_DIR}/system/training/phase1_segmentation/checkpoints', exist_ok=True)
shutil.copy(
    f'{DRIVE_DATA_DIR}/checkpoints_phase1/backbone_best.pt',
    f'{PROJECT_DIR}/system/training/phase1_segmentation/checkpoints/',
)

os.makedirs(f'{PROJECT_DIR}/system/training/phase2_warmup/checkpoints', exist_ok=True)
for fold in range(5):
    shutil.copy(
        f'{DRIVE_DATA_DIR}/checkpoints_phase2/fold{fold}_best.pt',
        f'{PROJECT_DIR}/system/training/phase2_warmup/checkpoints/',
    )

print(f'listo en {time.time()-t0:.0f}s')

## 5. Verificacion antes de entrenar

Confirma que el manifiesto, los 3 datasets y los 6 checkpoints de warm-start estan donde el codigo los espera - antes de lanzar horas de entrenamiento, no despues.

In [ ]:
import pandas as pd

manifest_path = f'{PROJECT_DIR}/datasets/manifest/rop_unified_manifest.parquet'
df = pd.read_parquet(manifest_path)
print('manifiesto:', len(df), 'filas,', df['dataset_origen'].value_counts().to_dict())

for origen in ['RIDIRP', 'FARFUM', 'ROP-VL']:
    base = f'{PROJECT_DIR}/datasets/extracted'
    rows = df[df['dataset_origen'] == origen]
    missing = sum(1 for p in rows['image_path'].sample(min(200, len(rows)), random_state=0)
                  if not os.path.exists(f'{base}/{p}'))
    print(f'{origen}: 0 de 200 muestras con path faltante' if missing == 0 else f'{origen}: FALTAN {missing}/200 - revisar la copia')

phase1_ckpt = f'{PROJECT_DIR}/system/training/phase1_segmentation/checkpoints/backbone_best.pt'
assert os.path.exists(phase1_ckpt), 'Falta el checkpoint de Fase 1'
print('Fase 1 checkpoint: OK')

for fold in range(5):
    p = f'{PROJECT_DIR}/system/training/phase2_warmup/checkpoints/fold{fold}_best.pt'
    assert os.path.exists(p), f'Falta el checkpoint de Fase 2, fold {fold}'
print('Fase 2 checkpoints (5 folds): OK')

print('\nTodo listo para entrenar.')

## 6. Entrenar - mismo `train.py` que local, sin modificar

`BATCH_SIZE=8` y `MAX_BAG_SIZE=8` (en `dataset.py`) quedan tal cual estan en el repo - no hace falta tocarlos aca, la GPU de Colab tiene mas VRAM que los 8GB locales que forzaron a reducirlos. `NUM_WORKERS` se recalcula solo segun los nucleos que de verdad tenga esta maquina de Colab (misma logica que local).

Esto corre los 5 folds completos (puede tardar). **Si Colab se desconecta a mitad de camino:** reconectate y volve a correr las celdas desde el paso 1 (montar Drive) - el paso 2 ahora actualiza el repo con `git pull` en vez de borrarlo (ya no se pierden los checkpoints locales que todavia no copiaste a Drive), el paso 4 se salta los datasets que ya estaban descomprimidos, y esta celda retoma donde quedo (`train.py` reanuda folds que ya tienen checkpoint guardado, no los reentrena de cero). Igual conviene correr el paso 7 periodicamente, no solo al final, por si la sesion de Colab en si se reinicia del todo (no solo se desconecta) y borra el disco local.

In [ ]:
%cd {PROJECT_DIR}/system/training/phase3_finetune
!python -u train.py

## 7. Copiar los checkpoints resultantes de vuelta a Drive

**Importante:** correr esto apenas termine el entrenamiento (o periodicamente mientras corre, por las dudas de una desconexion) - el disco local de Colab se borra al cerrar la sesion, y con el, los checkpoints si no se copiaron antes.

In [ ]:
src = f'{PROJECT_DIR}/system/training/phase3_finetune/checkpoints'
dst = f'{DRIVE_DATA_DIR}/checkpoints_phase3'

if not os.path.isdir(src) or not os.listdir(src):
    print(f'Todavia no hay ningun checkpoint de Fase 3 en {src} - esto es normal si el fold 0 '
          'todavia no termino su primera epoca (el checkpoint se guarda en silencio, sin aviso, '
          'apenas se imprime la linea "[fold 0][epoca 01/20] ..." en el paso 6). Esperar a esa '
          'primera linea antes de correr esta celda.')
else:
    os.makedirs(dst, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print('Checkpoints de Fase 3 copiados a:', dst)
    print(os.listdir(dst))